# 第二课拓展实验

三个独立实验：学习 BPE 规则、更新 embedding 表、验证二维旋转点积。无需安装额外依赖。

## 统计语料，学习合并规则

In [1]:
"""统计相邻片段频次，学习四条 BPE 合并规则。仅处理本例字符集合。"""
from collections import Counter

def merge_pair(parts, pair):
    result = []
    i = 0
    while i < len(parts):
        if tuple(parts[i:i + 2]) == pair:
            result.append(parts[i] + parts[i + 1])
            i += 2
        else:
            result.append(parts[i])
            i += 1
    return result

corpus = {"我喜欢喝牛奶": 5, "你喜欢喝牛奶": 3, "我喜欢喝酒": 2}
pieces = {word: list(word) for word in corpus}
learned_merges = []
for step in range(4):
    counts = Counter()
    for word, parts in pieces.items():
        for pair in zip(parts, parts[1:]):
            counts[pair] += corpus[word]
    best = max(counts, key=counts.get)  # 同频时使用首次出现的 pair，固定本例结果。
    learned_merges.append(best)
    print("第", step + 1, "轮：", best, "出现", counts[best], "次")
    pieces = {word: merge_pair(parts, best) for word, parts in pieces.items()}
print("学到的规则：", learned_merges)
print("切分结果：", pieces)


第 1 轮： ('喜', '欢') 出现 10 次
第 2 轮： ('喜欢', '喝') 出现 10 次
第 3 轮： ('喜欢喝', '牛') 出现 8 次
第 4 轮： ('喜欢喝牛', '奶') 出现 8 次
学到的规则： [('喜', '欢'), ('喜欢', '喝'), ('喜欢喝', '牛'), ('喜欢喝牛', '奶')]
切分结果： {'我喜欢喝牛奶': ['我', '喜欢喝牛奶'], '你喜欢喝牛奶': ['你', '喜欢喝牛奶'], '我喜欢喝酒': ['我', '喜欢喝', '酒']}


## 只更新 embedding 的分类实验

In [2]:
"""四个 token 的二分类：只训练 embedding，输出层固定为恒等映射。"""
import math

tokens = ["酒", "牛奶", "苹果", "香蕉"]
targets = [0, 0, 1, 1]  # 标签 0：配“喝”；标签 1：配“吃”。
table = [[0.1, -0.1], [-0.1, 0.1], [0.2, 0.0], [0.0, 0.2]]

def probabilities(row):
    values = [math.exp(v - max(row)) for v in row]
    return [v / sum(values) for v in values]

def loss():
    return sum(-math.log(probabilities(row)[target])
               for row, target in zip(table, targets)) / len(tokens)

initial_loss = loss()
print("更新前：", table)
for step in range(40):
    for row, target in zip(table, targets):
        probs = probabilities(row)
        for dimension in range(2):
            gradient = probs[dimension] - int(dimension == target)
            row[dimension] -= 0.2 * gradient
print("更新后：")
for token, row in zip(tokens, table):
    print(token, [round(v, 3) for v in row])
print("平均交叉熵：", round(initial_loss, 4), "→", round(loss(), 4))
print("只说明这四个 token 的分类拟合改善，不是完整语言模型预训练。")


更新前： [[0.1, -0.1], [-0.1, 0.1], [0.2, 0.0], [0.0, 0.2]]
更新后：
酒 [1.358, -1.358]
牛奶 [1.333, -1.333]
苹果 [-1.233, 1.433]
香蕉 [-1.258, 1.458]
平均交叉熵： 0.6981 → 0.0656
只说明这四个 token 的分类拟合改善，不是完整语言模型预训练。


## 保持相对位移，比较 RoPE 点积

In [3]:
"""二维旋转示例：Q/K 的旋转点积只依赖相对位移。"""
import math

def rotate(vector, position, frequency=0.2):
    angle = position * frequency
    x, y = vector
    return [x * math.cos(angle) - y * math.sin(angle),
            x * math.sin(angle) + y * math.cos(angle)]

def score(query_position, key_position):
    query = rotate([1.0, 2.0], query_position)
    key = rotate([3.0, 4.0], key_position)
    return sum(q * k for q, k in zip(query, key))

a, b = score(1, 3), score(6, 8)
print("Q 在 1、K 在 3：", round(a, 6))
print("Q 在 6、K 在 8：", round(b, 6))
print("相对位移同为 2，分数相同吗？", math.isclose(a, b, abs_tol=1e-12))
print("这是旋转点积的局部性质，不代表完整模型可无限外推。")


Q 在 1、K 在 3： 10.910508
Q 在 6、K 在 8： 10.910508
相对位移同为 2，分数相同吗？ True
这是旋转点积的局部性质，不代表完整模型可无限外推。
